In [1]:
import os
import pandas as pd
import joblib

# ============================================================
# ROBUST PROJECT ROOT DETECTION (PAPERMILL SAFE)
# ============================================================

def find_project_root():
    current = os.getcwd()

    while True:
        if (
            os.path.exists(os.path.join(current, "data")) and
            os.path.exists(os.path.join(current, "reports")) and
            os.path.exists(os.path.join(current, "models"))
        ):
            return current

        parent = os.path.dirname(current)

        if parent == current:
            raise RuntimeError("❌ Project root not found")

        current = parent


BASE_DIR = find_project_root()

print("✅ BASE_DIR:", BASE_DIR)

# ============================================================
# PATH DEFINITIONS
# ============================================================

DATA_PATH = os.path.join(BASE_DIR, "data", "real", "df_working.csv")
OUTPUT_PATH = os.path.join(BASE_DIR, "reports", "semantic_reports", "semantic_model_predictions.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "semantic_model.pkl")

# ============================================================
# DEBUG PATHS (CRITICAL)
# ============================================================

print("DATA_PATH:", DATA_PATH)
print("EXISTS:", os.path.exists(DATA_PATH))

# ============================================================
# FAIL-FAST
# ============================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"df_working not found at {DATA_PATH}")

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

if df.empty:
    raise ValueError("df_working is empty")

print("✅ Data loaded")
print("Columns:", list(df.columns))
print("✅ BASE_DIR:", BASE_DIR)

DATA_PATH = os.path.join(BASE_DIR, "data", "real", "df_working.csv")
OUTPUT_PATH = os.path.join(BASE_DIR, "reports", "semantic_reports", "semantic_model_predictions.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "semantic_model.pkl")

✅ BASE_DIR: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair
DATA_PATH: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/real/df_working.csv
EXISTS: True
✅ Data loaded
Columns: ['id', 'PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
✅ BASE_DIR: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair


In [2]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"df_working not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

if df.empty:
    raise ValueError("df_working is empty")

print("✅ Data loaded")
print("Columns:", list(df.columns))

✅ Data loaded
Columns: ['id', 'PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [3]:
model_bundle = None

if os.path.exists(MODEL_PATH):
    model_bundle = joblib.load(MODEL_PATH)
    print("✅ Semantic model loaded")
else:
    print("⚠️ No model found — proceeding rule-based only")

✅ Semantic model loaded


In [4]:
def build_features(series):

    series = series.astype(str)

    return {
        "pct_numeric": series.str.match(r'^-?\d+(\.\d+)?$').mean(),
        "pct_alpha": series.str.match(r'^[A-Za-z]+$').mean(),
        "pct_missing": series.isin(["", "?", "nan", "None"]).mean(),
        "avg_length": series.str.len().mean(),
        "nunique": series.nunique(),
        "has_email": int(series.str.contains("@").any()),
        "has_date": int(series.str.contains(r'\d{4}-\d{2}-\d{2}').any()),
        "has_numeric_range": int(series.str.contains(r'\d+').any()),

        # compatibility with training
        "numeric_ratio": series.str.match(r'^-?\d+(\.\d+)?$').mean(),
        "alpha_ratio": series.str.match(r'^[A-Za-z]+$').mean(),
        "missing_ratio": series.isin(["", "?", "nan", "None"]).mean(),
        "unique_count": series.nunique()
    }

In [5]:
def rule_based_type(col, series):
    col_lower = col.lower()

    # identifier
    if "id" in col_lower:
        return "identifier", 0.9

    # email
    if series.astype(str).str.contains("@").any():
        return "email", 0.9

    # numeric
    numeric_ratio = series.astype(str).str.match(r'^-?\d+(\.\d+)?$').mean()
    if numeric_ratio > 0.9:
        return "numeric", 0.9

    # categorical
    if series.nunique() < 50:
        return "categorical", 0.7

    return "unknown", 0.3

In [6]:
records = []

for col in df.columns:

    series = df[col]

    # ---------------------------
    # RULE-BASED (PRIMARY)
    # ---------------------------
    predicted_type, confidence = rule_based_type(col, series)

    # ---------------------------
    # MODEL ASSIST (SECONDARY)
    # ---------------------------
    if model_bundle is not None:

        features = build_features(series)
        feature_df = pd.DataFrame([features])

        # align with training features
        feature_df = feature_df.reindex(
            columns=model_bundle["features"],
            fill_value=0
        )

        model_pred = model_bundle["model"].predict(feature_df)[0]

        # only assist if low confidence
        if confidence < 0.6:
            predicted_type = model_pred
            confidence = 0.6

    # ---------------------------
    # SAVE RESULT
    # ---------------------------
    records.append({
        "column_name": col,
        "predicted_type": predicted_type,
        "confidence": confidence
    })

semantic_predictions = pd.DataFrame(records)

print("✅ Semantic predictions generated")
print(semantic_predictions)

✅ Semantic predictions generated
    column_name predicted_type  confidence
0            id     identifier         0.9
1   PassengerId     identifier         0.9
2      Survived        numeric         0.9
3        Pclass        numeric         0.9
4          Name           name         0.6
5           Sex    categorical         0.7
6           Age        numeric         0.6
7         SibSp        numeric         0.9
8         Parch        numeric         0.9
9        Ticket         salary         0.6
10         Fare        numeric         0.9
11        Cabin    categorical         0.6
12     Embarked    categorical         0.7


In [7]:
required_cols = ["column_name", "predicted_type", "confidence"]

for col in required_cols:
    if col not in semantic_predictions.columns:
        raise ValueError(f"Missing column: {col}")

if semantic_predictions.isnull().any().any():
    raise ValueError("Null values found in semantic_predictions")

semantic_predictions = semantic_predictions.sort_values(
    by=["column_name"]
).reset_index(drop=True)

print("✅ Validation passed")

✅ Validation passed


In [8]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

semantic_predictions.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved to {OUTPUT_PATH}")

✅ Saved to /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/semantic_reports/semantic_model_predictions.csv
